# 6 · Forecasting with Machine Learning (Multistep Strategies)

**Define the task first:**
- **Forecast origin**: the last time with data. Everything up to it can be used as a feature.
- **Forecast horizon**: the times being predicted. Store Sales needs 16 days.
- **Lead time**: the number of steps from the origin to the first horizon step. Store Sales has 1.

**Build the dataset:** each row is one forecast. Its features are the lags at the origin, and its targets are all H steps of the horizon.

| Strategy | Idea | sklearn |
|---|---|---|
| **MultiOutput** | one model with H outputs | `LinearRegression()` |
| **Direct** | one model per step | `MultiOutputRegressor(model)` |
| **Recursive** | one 1-step model, with its predictions fed back in as lags | a short loop |
| **DirRec** | one model per step, each also using the earlier steps' predictions | `RegressorChain(model)` |

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8-whitegrid")
plt.rc("figure", autolayout=True, figsize=(11, 4))
plot_params = dict(color="0.75", style=".-", markeredgecolor="0.25", markerfacecolor="0.25", legend=False)

DATA = Path("../input/store-sales-time-series-forecasting")   # change to where you unzipped the data

In [ ]:
def make_lags(ts, lags, lead_time=1):
    return pd.concat({f"y_lag_{i}": ts.shift(i) for i in range(lead_time, lags + lead_time)}, axis=1)


def make_multistep_target(ts, steps):
    return pd.concat({f"y_step_{i + 1}": ts.shift(-i) for i in range(steps)}, axis=1)


def plot_multistep(y, every=1, ax=None):
    """Draw each row of a multistep prediction as its own forecast path."""
    if ax is None:
        _, ax = plt.subplots()
    for date, preds in y[::every].iterrows():
        preds.index = pd.period_range(start=date, periods=len(preds))
        preds.plot(ax=ax)
    return ax

In [ ]:
store_sales = pd.read_csv(
    DATA / "train.csv",
    usecols=["store_nbr", "family", "date", "sales", "onpromotion"],
    dtype={"store_nbr": "category", "family": "category", "sales": "float32", "onpromotion": "uint32"},
    parse_dates=["date"],
)
store_sales["date"] = store_sales.date.dt.to_period("D")
store_sales = store_sales.set_index(["store_nbr", "family", "date"]).sort_index()

average_sales = store_sales.groupby("date").mean()["sales"].loc["2016":]

## 1. Build the multistep dataset
16-step horizon, 1-step lead time, 7 lag features.

In [ ]:
y = average_sales.copy()

X = make_lags(y, lags=7).dropna()
y = make_multistep_target(y, steps=16).dropna()
y, X = y.align(X, join="inner", axis=0)     # keep rows that have both features and all targets

pd.concat({"Targets": y, "Features": X}, axis=1).head()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, shuffle=False)

def rmse(a, b):
    return mean_squared_error(a, b) ** 0.5

## 2. MultiOutput: linear regression predicts all 16 steps at once

In [ ]:
from sklearn.linear_model import LinearRegression

model = LinearRegression().fit(X_train, y_train)
y_pred = pd.DataFrame(model.predict(X_test), index=X_test.index, columns=y.columns)
print(f"MultiOutput test RMSE: {rmse(y_test, y_pred):.2f}")

ax = average_sales[y_pred.index].plot(**plot_params)
plot_multistep(y_pred, every=16, ax=ax);

## 3. Direct: one XGBoost per step

In [ ]:
from sklearn.multioutput import MultiOutputRegressor
from xgboost import XGBRegressor

model = MultiOutputRegressor(XGBRegressor()).fit(X_train, y_train)
y_pred = pd.DataFrame(model.predict(X_test), index=X_test.index, columns=y.columns)
print(f"Direct test RMSE: {rmse(y_test, y_pred):.2f}")

## 4. DirRec: each step's model also sees the previous steps' predictions

In [ ]:
from sklearn.multioutput import RegressorChain

model = RegressorChain(XGBRegressor()).fit(X_train, y_train)
y_pred = pd.DataFrame(model.predict(X_test), index=X_test.index, columns=y.columns)
print(f"DirRec test RMSE: {rmse(y_test, y_pred):.2f}")

## 5. Recursive: one 1-step model, with its predictions fed back in as lags

In [ ]:
one_step = XGBRegressor().fit(X_train, y_train["y_step_1"])

def recursive_forecast(model, lags, steps):
    """lags: [lag_1, lag_2, ...], most recent first."""
    lags = list(lags)
    preds = []
    for _ in range(steps):
        y_hat = model.predict(pd.DataFrame([lags], columns=X.columns))[0]
        preds.append(y_hat)
        lags = [y_hat] + lags[:-1]          # newest prediction becomes lag_1
    return preds

y_pred = pd.DataFrame(
    [recursive_forecast(one_step, row, steps=16) for row in X_test.values],
    index=X_test.index, columns=y.columns,
)
print(f"Recursive test RMSE: {rmse(y_test, y_pred):.2f}")

## Store Sales: DirRec on all product families
Use the same recipe on many series: stack wide to long and label-encode the series id.

In [ ]:
family_sales = (
    store_sales
    .groupby(["family", "date"], observed=True).mean()
    .unstack("family")
    .loc["2017"]
)
y = family_sales.loc[:, "sales"]

X = make_lags(y, lags=4).dropna()
y = make_multistep_target(y, steps=16).dropna()
y, X = y.align(X, join="inner", axis=0)

X = X.stack("family").reset_index("family")     # wide → long
X["family"], families = X["family"].factorize()
y = y.stack("family")

model = RegressorChain(XGBRegressor()).fit(X, y)
y_pred = pd.DataFrame(model.predict(X), index=y.index, columns=y.columns).clip(0.0)

In [ ]:
FAMILY = "BEAUTY"
y_pred_ = y_pred.xs(FAMILY, level="family").loc["2017-04-01":]
ax = family_sales.loc["2017-04-01":, ("sales", FAMILY)].plot(**plot_params, alpha=0.5)
plot_multistep(y_pred_, ax=ax, every=16)
ax.set_title(f"{FAMILY}: 16-step forecasts");

**Takeaways**
- Define the origin, horizon and lead time before building any features.
- Direct gives each step its own model. Recursive and DirRec reuse predictions, so their errors grow with the horizon.
- The same multistep dataset works for many series once it's stacked to long format.